In [22]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import ttest_ind, chi2_contingency, pearsonr
import warnings
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
plt.style.use('default')
sns.set_palette("husl")

warnings.filterwarnings('ignore')


# Airbnb vs Traditional Hotels: A Comparative Analysis

## Project Overview
This analysis compares two major accommodation platforms to identify specific pricing strategies, 
booking behaviors, and market positioning differences.

## Data Sources
1. **Hotel Booking Demand Dataset**
   - Source: https://www.kaggle.com/datasets/jessemostipak/hotel-booking-demand
   
2. **Airbnb Open Data**
   - Source: https://www.kaggle.com/datasets/arianazmoudeh/airbnbopendata


### Dataset Overview

In [4]:
print("="*70)
print("LOADING DATASET 1: HOTEL BOOKINGS")
print("="*70)

df_hotel = pd.read_csv('data/hotel_bookings.csv')

print(f"\nDataset Shape: {df_hotel.shape}")
print(f"Rows: {df_hotel.shape[0]:,}")
print(f"Columns: {df_hotel.shape[1]}")

print(f"\nColumn Names:")
for i, col in enumerate(df_hotel.columns, 1):
    print(f"{i}. {col}")

print("\n" + "="*70)

LOADING DATASET 1: HOTEL BOOKINGS

Dataset Shape: (119390, 32)
Rows: 119,390
Columns: 32

Column Names:
1. hotel
2. is_canceled
3. lead_time
4. arrival_date_year
5. arrival_date_month
6. arrival_date_week_number
7. arrival_date_day_of_month
8. stays_in_weekend_nights
9. stays_in_week_nights
10. adults
11. children
12. babies
13. meal
14. country
15. market_segment
16. distribution_channel
17. is_repeated_guest
18. previous_cancellations
19. previous_bookings_not_canceled
20. reserved_room_type
21. assigned_room_type
22. booking_changes
23. deposit_type
24. agent
25. company
26. days_in_waiting_list
27. customer_type
28. adr
29. required_car_parking_spaces
30. total_of_special_requests
31. reservation_status
32. reservation_status_date



In [6]:
print("="*70)
print("LOADING DATASET 2: AIRBNB LISTINGS")
print("="*70)

df_airbnb = pd.read_csv('data/airbnb__data.csv')

print(f"\nDataset Shape: {df_airbnb.shape}")
print(f"Rows: {df_airbnb.shape[0]:,}")
print(f"Columns: {df_airbnb.shape[1]}")

print(f"\nColumn Names:")
for i, col in enumerate(df_airbnb.columns, 1):
    print(f"{i}. {col}")

print("\n" + "="*70)

LOADING DATASET 2: AIRBNB LISTINGS

Dataset Shape: (102599, 26)
Rows: 102,599
Columns: 26

Column Names:
1. id
2. NAME
3. host id
4. host_identity_verified
5. host name
6. neighbourhood group
7. neighbourhood
8. lat
9. long
10. country
11. country code
12. instant_bookable
13. cancellation_policy
14. room type
15. Construction year
16. price
17. service fee
18. minimum nights
19. number of reviews
20. last review
21. reviews per month
22. review rate number
23. calculated host listings count
24. availability 365
25. house_rules
26. license



In [7]:
print("="*70)
print("MISSING VALUES ANALYSIS - HOTELS")
print("="*70)

missing_hotel = df_hotel.isnull().sum()
missing_pct = (missing_hotel / len(df_hotel) * 100).round(2)

missing_df = pd.DataFrame({
    'Column': missing_hotel.index,
    'Missing_Count': missing_hotel.values,
    'Percentage': missing_pct.values
})

missing_df = missing_df[missing_df['Missing_Count'] > 0].sort_values('Percentage', ascending=False)

if len(missing_df) > 0:
    print("\nColumns with missing values:\n")
    print(missing_df.to_string(index=False))
else:
    print("\nNo missing values detected!")

MISSING VALUES ANALYSIS - HOTELS

Columns with missing values:

  Column  Missing_Count  Percentage
 company         112593       94.31
   agent          16340       13.69
 country            488        0.41
children              4        0.00


In [32]:
print("="*70)
print("MISSING VALUES ANALYSIS - AIRBNB")
print("="*70)

missing_airbnb = df_airbnb.isnull().sum()
missing_pct_ab = (missing_airbnb / len(df_airbnb) * 100).round(2)

missing_df_ab = pd.DataFrame({
    'Column': missing_airbnb.index,
    'Missing_Count': missing_airbnb.values,
    'Percentage': missing_pct_ab.values
})

missing_df_ab = missing_df_ab[missing_df_ab['Missing_Count'] > 0].sort_values('Percentage', ascending=False)

if len(missing_df_ab) > 0:
    print("\nTop 20 Columns with missing values:\n")
    print(missing_df_ab.head(20).to_string(index=False))
else:
    print("\nNo missing values detected!")

MISSING VALUES ANALYSIS - AIRBNB

Top 20 Columns with missing values:

                        Column  Missing_Count  Percentage
                   last review          15867       15.50
             reviews per month          15852       15.49
            country_normalized            527        0.51
                       country            527        0.51
              availability 365            448        0.44
                minimum nights            409        0.40
                     host name            401        0.39
            review rate number            326        0.32
calculated host listings count            319        0.31
        host_identity_verified            285        0.28
                          NAME            247        0.24
                   service fee            239        0.23
             Construction year            210        0.21
             number of reviews            183        0.18
                  country code            126        0.12
 

In [33]:
# Understanding Data Structure for Fair Comparison
print("="*70)
print("DATASET STRUCTURE ANALYSIS")
print("="*70)

print("\nHOTEL DATASET KEY COLUMNS:")
hotel_cols = df_hotel.columns.tolist()
print(f"Total columns: {len(hotel_cols)}")
print("\nRelevant columns for analysis:")
for col in hotel_cols:
    if any(keyword in col.lower() for keyword in ['country', 'adr', 'price', 'rate', 'guest', 'night', 'market', 'customer']):
        print(f"  - {col}")

print("\n" + "-"*70)

print("\nAIRBNB DATASET KEY COLUMNS:")
airbnb_cols = df_airbnb.columns.tolist()
print(f"Total columns: {len(airbnb_cols)}")
print("\nRelevant columns for analysis:")
for col in airbnb_cols[:20]:  # Show first 20 to avoid clutter
    print(f"  - {col}")

print("\n" + "="*70)

DATASET STRUCTURE ANALYSIS

HOTEL DATASET KEY COLUMNS:
Total columns: 32

Relevant columns for analysis:
  - stays_in_weekend_nights
  - stays_in_week_nights
  - country
  - market_segment
  - is_repeated_guest
  - customer_type
  - adr
  - country_normalized

----------------------------------------------------------------------

AIRBNB DATASET KEY COLUMNS:
Total columns: 25

Relevant columns for analysis:
  - id
  - NAME
  - host id
  - host_identity_verified
  - host name
  - neighbourhood group
  - neighbourhood
  - lat
  - long
  - country
  - country code
  - instant_bookable
  - cancellation_policy
  - room type
  - Construction year
  - price
  - service fee
  - minimum nights
  - number of reviews
  - last review



### FIND COMMON PATTERNS FOR COMPARATIVE ANALYSIS

#### Rating Analyses

In [40]:
# Find Rating Columns
print("="*70)
print("RATING COLUMN IDENTIFICATION")
print("="*70)

# Hotel ratings
hotel_rating_cols = [col for col in df_hotel.columns if any(keyword in col.lower() for keyword in ['rating', 'review', 'score', 'satisfaction'])]
print(f"\nHOTEL DATASET - Potential rating columns:")
if hotel_rating_cols:
    for col in hotel_rating_cols:
        print(f"  - {col}: {df_hotel[col].dtype}, non-null: {df_hotel[col].notna().sum():,}")
else:
    print("  No rating columns found")

# Airbnb ratings
airbnb_rating_cols = [col for col in df_airbnb.columns if any(keyword in col.lower() for keyword in ['rating', 'review', 'score', 'satisfaction'])]
print(f"\nAIRBNB DATASET - Potential rating columns:")
if airbnb_rating_cols:
    for col in airbnb_rating_cols[:10]:  # Limit output
        print(f"  - {col}: {df_airbnb[col].dtype}, non-null: {df_airbnb[col].notna().sum():,}")
else:
    print("  No rating columns found")

# Select best rating columns
hotel_rating = None
if hotel_rating_cols:
    # Prefer columns with most non-null values
    hotel_rating = max(hotel_rating_cols, key=lambda x: df_hotel[x].notna().sum())
    print(f"\nSelected HOTEL rating column: '{hotel_rating}'")
else:
    print("\nNo hotel rating column available")

airbnb_rating = None
rating_options = ['review_scores_rating', 'review scores rating', 'rating', 'Rating']
for col in rating_options:
    if col in df_airbnb.columns:
        airbnb_rating = col
        print(f"Selected AIRBNB rating column: '{col}'")
        break

if not airbnb_rating and airbnb_rating_cols:
    airbnb_rating = max(airbnb_rating_cols, key=lambda x: df_airbnb[x].notna().sum())
    print(f"Selected AIRBNB rating column: '{airbnb_rating}'")
elif not airbnb_rating:
    print("No Airbnb rating column available")

# Final conclusion
print("\n" + "="*70)
print("RATING COMPARISON FEASIBILITY")
print("="*70)

if hotel_rating and airbnb_rating:
    print("\nSTATUS: Rating comparison is POSSIBLE")
    print(f"  Hotel rating: '{hotel_rating}'")
    print(f"  Airbnb rating: '{airbnb_rating}'")
    print("\nWe can proceed with rating-based comparative analysis")
elif not hotel_rating and not airbnb_rating:
    print("\nSTATUS: Rating comparison is NOT POSSIBLE")
    print("REASON: Neither dataset contains rating columns")
    print("\nCONCLUSION:")
    print("Rating-based comparison cannot be performed with these datasets.")
    print("Alternative analyses available:")
    print("  - Price analysis (Airbnb)")
    print("  - Booking patterns (Hotels)")
    print("  - Geographic distribution")
    print("  - Temporal trends")
elif not hotel_rating:
    print("\nSTATUS: Rating comparison is NOT POSSIBLE")
    print("REASON: Hotel dataset lacks rating columns")
    print(f"  Airbnb has: '{airbnb_rating}'")
    print("\nCONCLUSION:")
    print("Cannot perform rating comparison as hotel ratings are unavailable.")
    print("Focus will remain on available metrics in each dataset independently.")
else:  # not airbnb_rating
    print("\nSTATUS: Rating comparison is NOT POSSIBLE")
    print("REASON: Airbnb dataset lacks rating columns")
    print(f"  Hotel has: '{hotel_rating}'")
    print("\nCONCLUSION:")
    print("Cannot perform rating comparison as Airbnb ratings are unavailable.")
    print("Analysis will focus on price patterns and booking behaviors separately.")

print("="*70)

RATING COLUMN IDENTIFICATION

HOTEL DATASET - Potential rating columns:
  No rating columns found

AIRBNB DATASET - Potential rating columns:
  - number of reviews: float64, non-null: 102,169
  - last review: object, non-null: 86,485
  - reviews per month: float64, non-null: 86,500
  - review rate number: float64, non-null: 102,026

No hotel rating column available
Selected AIRBNB rating column: 'number of reviews'

RATING COMPARISON FEASIBILITY

STATUS: Rating comparison is NOT POSSIBLE
REASON: Hotel dataset lacks rating columns
  Airbnb has: 'number of reviews'

CONCLUSION:
Cannot perform rating comparison as hotel ratings are unavailable.
Focus will remain on available metrics in each dataset independently.
